# A replication of methods used in the AlphaGo paper "Mastering the game of Go with deep neural networks and tree search

In [ ]:
!pip install -q line_profiler
%load_ext line_profiler


In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Game Environment

A Python-based game env taken from https://github.com/maxpumperla/deep_learning_and_the_game_of_go


### Point, Player

In [2]:
import enum
from collections import namedtuple

class Point(namedtuple('Point', 'row col')):
    def neighbors(self):
        return [
            Point(self.row - 1, self.col),
            Point(self.row + 1, self.col),
            Point(self.row, self.col - 1),
            Point(self.row, self.col + 1),
        ]
class Player(enum.Enum):
    black = 1
    white = 2

    @property
    def other(self):
        return Player.black if self == Player.white else Player.white

### GameResult, Territory

In [3]:
from __future__ import absolute_import
from collections import namedtuple

class Territory(object):
    def __init__(self, territory_map):
        self.num_black_territory = 0
        self.num_white_territory = 0
        self.num_black_stones = 0
        self.num_white_stones = 0
        self.num_dame = 0
        self.dame_points = []
        for point, status in territory_map.items():
            if status == Player.black:
                self.num_black_stones += 1
            elif status == Player.white:
                self.num_white_stones += 1
            elif status == 'territory_b':
                self.num_black_territory += 1
            elif status == 'territory_w':
                self.num_white_territory += 1
            elif status == 'dame':
                self.num_dame += 1
                self.dame_points.append(point)

class GameResult(namedtuple('GameResult', 'b w komi')):
    @property
    def winner(self):
        if self.b > self.w + self.komi:
            return Player.black
        if self.b < self.w + self.komi:
            return Player.white
        return None

    @property
    def winning_margin(self):
        w = self.w + self.komi
        return abs(self.b - w)

    def __str__(self):
        w = self.w + self.komi
        if self.b > w:
            return 'B+%.1f' % (self.b - w,)
        return 'W+%.1f' % (w - self.b,)

def _collect_region(start_pos, board, visited=None):
    if visited is None:
        visited = {}
    if start_pos in visited:
        return [], set()
    all_points = [start_pos]
    all_borders = set()
    visited[start_pos] = True
    here = board.get(start_pos)
    deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    for delta_r, delta_c in deltas:
        next_p = Point(row=start_pos.row + delta_r, col=start_pos.col + delta_c)
        if not board.is_on_grid(next_p):
            continue
        neighbor = board.get(next_p)
        if neighbor == here:
            points, borders = _collect_region(next_p, board, visited)
            all_points += points
            all_borders |= borders
        else:
            all_borders.add(neighbor)
    return all_points, all_borders

def evaluate_territory(board):
    status = {}
    for r in range(1, board.num_rows + 1):
        for c in range(1, board.num_cols + 1):
            p = Point(row=r, col=c)
            if p in status:
                continue
            stone = board.get(p)
            if stone is not None:
                status[p] = board.get(p)
            else:
                group, neighbors = _collect_region(p, board)
                if len(neighbors) == 1:
                    neighbor_stone = neighbors.pop()
                    stone_str = 'b' if neighbor_stone == Player.black else 'w'
                    fill_with = 'terrtory_' + stone_str
                else:
                    fill_with = 'dame'
                for pos in group:
                    status[pos] = fill_with
    return Territory(status)

def compute_game_result(game_state):
    territory = evaluate_territory(game_state.board)
    return GameResult(
        territory.num_black_territory + territory.num_black_stones,
        territory.num_white_territory + territory.num_white_stones,
        komi=7.5)

### GameBoard, GameState, Move, GoString

In [4]:
import copy

class Board():
    def __init__(self, num_rows, num_cols):
        self.num_rows = num_rows
        self.num_cols = num_cols
        self._grid = {}
        self._hash = EMPTY_BOARD

    # replace string should be immutable, which means that it should return a new grid
    def _replace_string(self, new_string):
        for point in new_string.stones:
            self._grid[point] = new_string

    def _remove_string(self, string):
        for point in string.stones:
            for neighbor in point.neighbors():
                neighbor_string = self._grid.get(neighbor)
                if neighbor_string is None:
                    continue
                if neighbor_string is not string:
                    self._replace_string(neighbor_string.with_liberty(point))
            self._grid[point] = None
            self._hash ^= HASH_CODE[point, string.color]

    def zobrist_hash(self):
        return self._hash

    def place_stone(self, player, point):
        assert self.is_on_grid(point)
        assert self._grid.get(point) is None
        adjacent_same_color = []
        adjacent_opposite_color = []
        liberties = []
        for neighbor in point.neighbors():
            if not self.is_on_grid(neighbor):
                continue
            neighbor_string = self._grid.get(neighbor)
            if neighbor_string is None:
                liberties.append(neighbor)
            elif neighbor_string.color == player:
                if neighbor_string not in adjacent_same_color:
                    adjacent_same_color.append(neighbor_string)
            else:
                if neighbor_string not in adjacent_opposite_color:
                    adjacent_opposite_color.append(neighbor_string)
        new_string = GoString(player, [point], liberties)
        for same_color_string in adjacent_same_color:
            new_string = new_string.merged_with(same_color_string)
        for new_string_point in new_string.stones:
            self._grid[new_string_point] = new_string
        self._hash ^= HASH_CODE[point, player]
        for other_color_string in adjacent_opposite_color:
            replacement = other_color_string.without_liberty(point)
            if replacement.num_liberties:
                self._replace_string(other_color_string.without_liberty(point))
            else:
                self._remove_string(other_color_string)

    def is_on_grid(self, point):
        return 1 <= point.row <= self.num_rows and \
            1 <= point.col <= self.num_cols

    def get(self, point):
        string = self._grid.get(point)
        if string is None:
            return None
        return string.color

    def get_go_string(self, point):
        string = self._grid.get(point)
        if string is None:
            return None
        return string

class Move():
    def __init__(self, point=None, is_pass=False, is_resign=False):
        assert (point is not None) ^ is_pass ^ is_resign
        self.point = point
        self.is_play = (self.point is not None)
        self.is_pass = is_pass
        self.is_resign = is_resign

    @classmethod
    def play(cls, point):
        return Move(point=point)

    @classmethod
    def pass_turn(cls):
        return Move(is_pass=True)

    @classmethod
    def resign(cls):
        return Move(is_resign=True)

# Chain of connected stones (used to e.g. efficiently check for liberties)
class GoString():
    def __init__(self, color, stones, liberties):
        self.color = color
        self.stones = frozenset(stones)
        self.liberties = frozenset(liberties)

    def without_liberty(self, point):
        new_liberties = self.liberties - set([point])
        return GoString(self.color, self.stones, new_liberties)

    def with_liberty(self, point):
        new_liberties = self.liberties | set([point])
        return GoString(self.color, self.stones, new_liberties)

    def merged_with(self, go_string):
        assert go_string.color == self.color
        combined_stones = self.stones | go_string.stones
        return GoString(
            self.color,
            combined_stones,
            (self.liberties | go_string.liberties) - combined_stones
        )

    @property
    def num_liberties(self):
        return len(self.liberties)

    def __eq__(self, other):
        return isinstance(other, GoString) and \
            self.color == other.color and \
            self.stones == other.stones and \
            self.liberties == other.liberties

class GameState():
    def __init__(self, board, next_player, previous, move):
        self.board = board
        self.next_player = next_player
        self.previous_state = previous
        if self.previous_state is None:
            self.previous_states = frozenset()
        else:
            self.previous_states = frozenset(
                previous.previous_states |
                {(previous.next_player, previous.board.zobrist_hash())})
        self.last_move = move

    def apply_move(self, move):
        if move.is_play:
            next_board = copy.deepcopy(self.board)
            next_board.place_stone(self.next_player, move.point)
        else:
            next_board = self.board
        return GameState(next_board, self.next_player.other, self, move)

    def is_over(self):
        if self.last_move is None:
            return False
        if self.last_move.is_resign:
            return True
        second_last_move = self.previous_state.last_move
        if second_last_move is None:
            return False
        return self.last_move.is_pass and second_last_move.is_pass

    def is_move_self_capture(self, player, move):
        if not move.is_play:
            return False
        next_board = copy.deepcopy(self.board)
        next_board.place_stone(player, move.point)
        new_string = next_board.get_go_string(move.point)
        return new_string.num_liberties == 0

    def is_valid_move(self, move):
        if self.is_over():
            return False
        if move.is_pass or move.is_resign:
            return True
        return (
            self.board.get(move.point) is None and
            not self.is_move_self_capture(self.next_player, move) and
            not self.does_move_violate_ko(self.next_player, move))
    
    def legal_moves(self):
        if self.is_over():
            return []
        moves = []
        for row in range(1, self.board.num_rows + 1):
            for col in range(1, self.board.num_cols + 1):
                move = Move.play(Point(row, col))
                if self.is_valid_move(move):
                    moves.append(move)
        moves.append(Move.pass_turn())
        moves.append(Move.resign())
        return moves
    
    def winner(self):
        if not self.is_over():
            return None
        if self.last_move.is_resign:
            return self.next_player
        game_result = compute_game_result(self)
        return game_result.winner

    @classmethod
    def new_game(cls, board_size):
        if isinstance(board_size, int):
            board_size = (board_size, board_size)
        board = Board(*board_size)
        return GameState(board, Player.black, None, None)

    @property
    def situation(self):
        return (self.next_player, self.board)

    def does_move_violate_ko(self, player, move):
        if not move.is_play:
            return False
        next_board = copy.deepcopy(self.board)
        next_board.place_stone(player, move.point)
        next_situation = (player.other, next_board.zobrist_hash())
        return next_situation in self.previous_states

## Zobrist Hash

In [5]:
import random

MAX63 = 0x7fffffffffffffff

HASH_CODE = {}
EMPTY_BOARD = 0

for row in range(1, 20):
    for col in range(1, 20):
        for state in (1, 2):
            code = random.randint(0, MAX63)
            HASH_CODE[Point(row, col), state] = code

print(HASH_CODE)

{(Point(row=1, col=1), 1): 7237747715358997058, (Point(row=1, col=1), 2): 6508946310985845069, (Point(row=1, col=2), 1): 897649952695469243, (Point(row=1, col=2), 2): 4543384693301281806, (Point(row=1, col=3), 1): 6205168720838951032, (Point(row=1, col=3), 2): 2046342435617238626, (Point(row=1, col=4), 1): 9057335304903363928, (Point(row=1, col=4), 2): 4994924850819244813, (Point(row=1, col=5), 1): 1622422250417139415, (Point(row=1, col=5), 2): 4826155542915858682, (Point(row=1, col=6), 1): 8978968114042244441, (Point(row=1, col=6), 2): 3506219100693516890, (Point(row=1, col=7), 1): 4988955153273795834, (Point(row=1, col=7), 2): 7676172804390065842, (Point(row=1, col=8), 1): 4041182379891834614, (Point(row=1, col=8), 2): 882075358051584502, (Point(row=1, col=9), 1): 2208965117928719478, (Point(row=1, col=9), 2): 181629298963594760, (Point(row=1, col=10), 1): 1373817962402363285, (Point(row=1, col=10), 2): 5494491845083963457, (Point(row=1, col=11), 1): 4956986424056323312, (Point(row=1

## Fast Game Environment

In [6]:
import numpy as np
from collections import defaultdict

class FastBoard:
    """Optimized board using numpy arrays and Union-Find for group tracking"""
    def __init__(self, num_rows, num_cols):
        self.num_rows = num_rows
        self.num_cols = num_cols
        # Use numpy arrays: 0=empty, 1=black, 2=white
        self.grid = np.zeros((num_rows, num_cols), dtype=np.int8)
        self._hash = 0
        self._is_shallow = False
        
        # Union-Find data structures
        self.parent = {}  # Point -> Point (parent in union-find)
        self.liberties = {}  # Point (root) -> set of liberty Points
        self.stones = {}  # Point (root) -> set of stone Points
        self.color = {}  # Point -> player color (1 or 2)
        
    def copy(self):
        """Shallow copy for validation, deep copy only when modified"""
        new_board = FastBoard.__new__(FastBoard)
        new_board.num_rows = self.num_rows
        new_board.num_cols = self.num_cols
        new_board.grid = self.grid.copy()
        new_board._hash = self._hash
        
        # Shallow copy - share references (copy-on-write)
        new_board.parent = self.parent
        new_board.color = self.color
        new_board.liberties = self.liberties
        new_board.stones = self.stones
        new_board._is_shallow = True
        
        return new_board
    
    def _ensure_deep_copy(self):
        """Convert shallow copy to deep copy before modification"""
        if hasattr(self, '_is_shallow') and self._is_shallow:
            self.parent = self.parent.copy()
            self.color = self.color.copy()
            self.liberties = {k: v.copy() for k, v in self.liberties.items()}
            self.stones = {k: v.copy() for k, v in self.stones.items()}
            self._is_shallow = False
    
    def _find(self, point):
        """Find group root with path compression - O(α(n)) ≈ O(1)"""
        if point not in self.parent:
            return point
        # Path compression
        if self.parent[point] != point:
            self.parent[point] = self._find(self.parent[point])
        return self.parent[point]
    
    def _union(self, point1, point2):
        """Merge two groups - O(α(n)) ≈ O(1)"""
        root1 = self._find(point1)
        root2 = self._find(point2)
        
        if root1 == root2:
            return root1
        
        # Merge smaller into larger for better performance
        if len(self.stones[root1]) < len(self.stones[root2]):
            root1, root2 = root2, root1
        
        # Update parent pointer
        self.parent[root2] = root1
        
        # Merge stones and liberties
        self.stones[root1] |= self.stones[root2]
        self.liberties[root1] |= self.liberties[root2]
        
        # Remove stones from liberties (they're now occupied)
        self.liberties[root1] -= self.stones[root1]
        
        # Clean up old root
        del self.stones[root2]
        del self.liberties[root2]
        
        return root1
    
    def _get_neighbors(self, point):
        """Get valid neighbor points - O(1)"""
        row, col = point.row - 1, point.col - 1
        neighbors = []
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = row + dr, col + dc
            if 0 <= nr < self.num_rows and 0 <= nc < self.num_cols:
                neighbors.append(Point(nr + 1, nc + 1))
        return neighbors
    
    def place_stone(self, player, row, col):
        """Place stone and handle captures - O(1) amortized"""
        # Ensure we have a deep copy before modifying
        self._ensure_deep_copy()
        
        point = Point(row + 1, col + 1)
        
        # Update grid and hash
        self.grid[row, col] = player
        self._hash ^= HASH_CODE[point, player]
        
        # Initialize this stone as its own group
        self.color[point] = player
        self.parent[point] = point
        self.stones[point] = {point}
        
        # Initial liberties = empty neighbors
        self.liberties[point] = set()
        for neighbor in self._get_neighbors(point):
            neighbor_row, neighbor_col = neighbor.row - 1, neighbor.col - 1
            if self.grid[neighbor_row, neighbor_col] == 0:
                self.liberties[point].add(neighbor)
        
        # Merge with friendly neighboring groups
        my_root = point
        for neighbor in self._get_neighbors(point):
            if neighbor in self.color and self.color[neighbor] == player:
                my_root = self._union(my_root, neighbor)
        
        # Remove this point from opponent group liberties
        opponent = 3 - player
        for neighbor in self._get_neighbors(point):
            if neighbor in self.color and self.color[neighbor] == opponent:
                neighbor_root = self._find(neighbor)
                if neighbor_root in self.liberties:  # Check if group still exists
                    self.liberties[neighbor_root].discard(point)
        
        # Check for captures of opponent groups
        # Track which groups we've captured to avoid processing the same group twice
        captured_groups = set()
        for neighbor in self._get_neighbors(point):
            if neighbor in self.color and self.color[neighbor] == opponent:
                neighbor_root = self._find(neighbor)
                # Only process each group once
                if neighbor_root not in captured_groups and neighbor_root in self.liberties:
                    if len(self.liberties[neighbor_root]) == 0:
                        self._remove_group(neighbor_root)
                        captured_groups.add(neighbor_root)
                        
    def _remove_group(self, root):
        """Remove a captured group - O(group_size)"""
        stones_to_remove = self.stones[root].copy()
        
        for stone in stones_to_remove:
            stone_row, stone_col = stone.row - 1, stone.col - 1
            
            # Update grid and hash
            self.grid[stone_row, stone_col] = 0
            self._hash ^= HASH_CODE[stone, self.color[stone]]
            
            # Remove from color and parent tracking
            del self.color[stone]
            del self.parent[stone]
            
            # Add as liberty to neighboring groups (only if they still exist)
            for neighbor in self._get_neighbors(stone):
                if neighbor in self.color:
                    neighbor_root = self._find(neighbor)
                    # Check if this group still exists (might have been captured too)
                    if neighbor_root in self.liberties:
                        self.liberties[neighbor_root].add(stone)
        
        # Clean up group data
        del self.stones[root]
        del self.liberties[root]
    
    def count_liberties(self, row, col):
        """Count liberties of group at position - O(1)"""
        point = Point(row + 1, col + 1)
        if point not in self.color:
            return 0
        root = self._find(point)
        return len(self.liberties[root])
    
    def simulate_move_hash(self, player, row, col):
        """Calculate what the hash would be after placing a stone WITHOUT actually placing it
        This is much faster than copying the board"""
        point = Point(row + 1, col + 1)
        
        # Start with adding the new stone to hash
        new_hash = self._hash ^ HASH_CODE[point, player]
        
        # Check if this move would capture any opponent groups
        opponent = 3 - player
        for neighbor in self._get_neighbors(point):
            if neighbor in self.color and self.color[neighbor] == opponent:
                neighbor_root = self._find(neighbor)
                # Check if this opponent group has only 1 liberty (our move point)
                if len(self.liberties[neighbor_root]) == 1 and point in self.liberties[neighbor_root]:
                    # This group would be captured - remove from hash
                    for stone in self.stones[neighbor_root]:
                        new_hash ^= HASH_CODE[stone, opponent]
        
        return new_hash
    
    def zobrist_hash(self):
        """Get board hash - O(1)"""
        return self._hash
    
    def get(self, row, col):
        """Get stone at position - O(1)"""
        return self.grid[row, col]

class FastGameState:
    """Optimized game state"""
    def __init__(self, board, next_player, previous_hash=None, last_move=None, second_last_move=None):
        self.board = board
        self.next_player = next_player  # 1 for black, 2 for white
        self.previous_hashes = set()
        if previous_hash is not None:
            self.previous_hashes.add(previous_hash)
        self.last_move = last_move
        self.second_last_move = second_last_move
    
    def apply_move(self, move):
        """Apply move and return new state"""
        if move.is_play:
            point = move.point
            row = point.row - 1
            col = point.col - 1
            new_board = self.board.copy()
            new_board.place_stone(self.next_player, row, col)
        else:
            new_board = self.board.copy()
        
        new_state = FastGameState(
            new_board, 
            3 - self.next_player, 
            self.board.zobrist_hash(), 
            move, 
            self.last_move
        )
        new_state.previous_hashes = self.previous_hashes.copy()
        new_state.previous_hashes.add(self.board.zobrist_hash())
        
        return new_state

    def is_over(self):
        """Check if game is over"""
        if self.last_move is None:
            return False
        if self.last_move.is_resign:
            return True
        if self.second_last_move is None:
            return False
        return self.last_move.is_pass and self.second_last_move.is_pass
    
    def is_valid_move(self, move):
        """Check if move is valid - optimized with early rejection"""
        if self.is_over():
            return False
        if move.is_pass or move.is_resign:
            return True

        point = move.point
        row = point.row - 1
        col = point.col - 1

        # Early rejection: position occupied
        if self.board.get(row, col) != 0:
            return False
        
        # Fast check: if any neighbor is empty, this move has liberties
        # and cannot be self-capture (most common case)
        neighbors = self.board._get_neighbors(point)
        has_empty_neighbor = False
        opponent = 3 - self.next_player
        
        for neighbor in neighbors:
            neighbor_row, neighbor_col = neighbor.row - 1, neighbor.col - 1
            neighbor_color = self.board.get(neighbor_row, neighbor_col)
            
            if neighbor_color == 0:
                # Empty neighbor means move has liberties
                has_empty_neighbor = True
                break
        
        # If we have empty neighbor, only need to check ko
        if has_empty_neighbor:
            # Simulate hash without copying board!
            simulated_hash = self.board.simulate_move_hash(self.next_player, row, col)
            return simulated_hash not in self.previous_hashes
        
        # No empty neighbors - need full validation
        # Check if move captures opponent stones or connects to friendly group with liberties
        would_capture = False
        connects_to_group_with_liberties = False
        
        for neighbor in neighbors:
            neighbor_row, neighbor_col = neighbor.row - 1, neighbor.col - 1
            neighbor_color = self.board.get(neighbor_row, neighbor_col)
            
            if neighbor_color == opponent:
                # Check if this opponent group has only 1 liberty (our move would capture)
                if self.board.count_liberties(neighbor_row, neighbor_col) == 1:
                    would_capture = True
                    break
            elif neighbor_color == self.next_player:
                # Check if friendly group has >1 liberty (so we can connect safely)
                if self.board.count_liberties(neighbor_row, neighbor_col) > 1:
                    connects_to_group_with_liberties = True
        
        # If would capture OR connects to safe group, move is valid (not self-capture)
        if would_capture or connects_to_group_with_liberties:
            # Check ko without copying
            simulated_hash = self.board.simulate_move_hash(self.next_player, row, col)
            return simulated_hash not in self.previous_hashes
        
        # Otherwise it's self-capture
        return False
    
    def legal_moves(self):
        """Return all legal moves"""
        legal = []
        for row in range(self.board.num_rows):
            for col in range(self.board.num_cols):
                move = Move.play(Point(row + 1, col + 1))
                if self.is_valid_move(move):
                    legal.append(move)
        legal.append(Move.pass_turn())
        legal.append(Move.resign())
        return legal

    def winner(self):
        """Determine winner"""
        if not self.is_over():
            return None
        if self.last_move.is_resign:
            return self.next_player
        game_result = compute_game_result_fast(self, komi=0)
        return game_result.winner
    
    @classmethod
    def new_game(cls, board_size=19):
        """Create new game"""
        board = FastBoard(board_size, board_size)
        return FastGameState(board, 1)  # Black plays first

In [7]:
class FastTerritory:
    """Fast territory evaluation results"""
    __slots__ = ['num_black_territory', 'num_white_territory', 
                 'num_black_stones', 'num_white_stones', 
                 'num_dame', 'dame_points']
    
    def __init__(self):
        self.num_black_territory = 0
        self.num_white_territory = 0
        self.num_black_stones = 0
        self.num_white_stones = 0
        self.num_dame = 0
        self.dame_points = []

class FastGameResult:
    """Game result with score calculation"""
    def __init__(self, b, w, komi=7.5):
        self.b = b
        self.w = w
        self.komi = komi
    
    @property
    def winner(self):
        """Return 1 for black, 2 for white, 0 for tie"""
        if self.b > self.w + self.komi:
            return 1  # Black
        if self.b < self.w + self.komi:
            return 2  # White
        return 0  # Tie
    
    @property
    def winning_margin(self):
        w = self.w + self.komi
        return abs(self.b - w)
    
    def __str__(self):
        w = self.w + self.komi
        if self.b > w:
            return f'B+{self.b - w:.1f}'
        return f'W+{w - self.b:.1f}'

def evaluate_territory_fast(board):
    """Optimized territory evaluation using numpy and flood fill"""
    grid = board.grid
    rows, cols = grid.shape
    
    # Status: 0=unvisited, 1=black stone, 2=white stone, 
    # 3=black territory, 4=white territory, 5=dame
    status = np.zeros((rows, cols), dtype=np.int8)
    status[grid > 0] = grid[grid > 0]  # Copy stones
    
    territory = Territory()
    
    # Count stones directly from grid
    territory.num_black_stones = np.sum(grid == 1)
    territory.num_white_stones = np.sum(grid == 2)
    
    # Flood fill empty regions
    for r in range(rows):
        for c in range(cols):
            if status[r, c] == 0:  # Empty and unvisited
                points, borders = _collect_region_fast(r, c, grid, status)
                
                # Determine territory ownership
                if len(borders) == 1:
                    # Single color border = territory
                    owner = borders.pop()
                    if owner == 1:
                        territory.num_black_territory += len(points)
                        fill_value = 3
                    else:
                        territory.num_white_territory += len(points)
                        fill_value = 4
                else:
                    # Multiple colors or no border = dame
                    territory.num_dame += len(points)
                    territory.dame_points.extend(points)
                    fill_value = 5
                
                # Mark all points in region
                for pr, pc in points:
                    status[pr, pc] = fill_value
    
    return territory


def _collect_region_fast(start_r, start_c, grid, status):
    """Fast flood fill using stack instead of recursion"""
    rows, cols = grid.shape
    points = []
    borders = set()
    
    # Stack-based flood fill
    stack = [(start_r, start_c)]
    visited = np.zeros((rows, cols), dtype=bool)
    
    while stack:
        r, c = stack.pop()
        
        if visited[r, c]:
            continue
        visited[r, c] = True
        
        points.append((r, c))
        
        # Check all 4 neighbors
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            
            if not (0 <= nr < rows and 0 <= nc < cols):
                continue
            
            neighbor_value = grid[nr, nc]
            
            if neighbor_value == 0 and not visited[nr, nc]:
                # Empty space, continue flood fill
                stack.append((nr, nc))
            elif neighbor_value > 0:
                # Stone found, add to borders
                borders.add(neighbor_value)
    
    return points, borders


def compute_game_result_fast(game_state, komi=7.5):
    """Compute final game result"""
    territory = evaluate_territory_fast(game_state.board)
    
    black_score = territory.num_black_territory + territory.num_black_stones
    white_score = territory.num_white_territory + territory.num_white_stones
    
    return FastGameResult(black_score, white_score, komi)

## Fast Random Bot Self Play

This is to test the working of Fast Environment

In [24]:
%%prun -s cumulative
import random
import time
class FastRandomBot:
    def __init__(self, player):
        self.player = player
    def play_move(self, game_state):
        assert isinstance(game_state, FastGameState)
        legal_moves = game_state.legal_moves()
        random_move = random.choice(legal_moves)
        new_game_state = game_state.apply_move(random_move)
        return new_game_state, 3 - self.player

def play_game():
    game_state = FastGameState.new_game()
    
    agent = FastRandomBot(player = 1)
    opponent = FastRandomBot(player = 2)
    next_player = 1
    while not game_state.is_over():
        if next_player == 1:
            game_state, next_player = agent.play_move(game_state)
        else:
            game_state, next_player = opponent.play_move(game_state)
    return game_state.winner()

start = time.time()
for _ in range(5):
    winner = play_game()
    print(winner)
end = time.time()

print(f"time elapsed {end - start} seconds")


2
1
1
2
2
time elapsed 3.7822325229644775 seconds
 

         5725688 function calls (5711224 primitive calls) in 3.783 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    3.783    3.783 {built-in method builtins.exec}
        1    0.000    0.000    3.782    3.782 <string>:1(<module>)
        5    0.015    0.003    3.782    0.756 <string>:13(play_game)
      497    0.001    0.000    3.766    0.008 <string>:6(play_move)
      497    0.258    0.001    3.694    0.007 1796600014.py:314(legal_moves)
   179417    1.035    0.000    3.178    0.000 1796600014.py:250(is_valid_move)
   282376    0.998    0.000    1.571    0.000 1796600014.py:83(_get_neighbors)
   140032    0.293    0.000    1.194    0.000 1796600014.py:177(simulate_move_hash)
  1399197    0.346    0.000    0.613    0.000 <string>:1(<lambda>)
  1399694    0.267    0.000    0.267    0.000 {built-in method __new__ of type object at 0x93e380}
   179417    0.099    0.000    0.176    0.000 9299694

## Batched Game Environment

In [8]:
import torch

class BatchedGameState:
    def __init__(self, batch_size=16):
        self.batch_size = batch_size
        self.games = []
        self.batched_game_states = torch.empty(self.batch_size, 6, 19, 19)
        self.batched_action_mask = torch.empty(self.batch_size, 361)
        self.encoder = FastFourplaneEncoder()
        self.batched_rewards = torch.empty(self.batch_size)

    def generate_batched_input(self, game_states, legal_moves):
        for idx, game_state in enumerate(game_states):
            encoded_game_state = self.encoder.encode(game_state)
            encoded_game_tensor = torch.from_numpy(encoded_game_state)
            legal_moves_points = [self.encoder.encode_point(move.point) for move in legal_moves[idx] if move.is_play]
            action_mask = torch.zeros(361)
            action_mask[legal_moves_points] = 1.0
            self.batched_game_states[idx] = encoded_game_tensor
            self.batched_action_mask[idx] = action_mask

        self.batched_game_states = self.batched_game_states.to(torch.float32)
        self.batched_action_mask = self.batched_action_mask.to(torch.float32)
        return self.batched_game_states.to(device), self.batched_action_mask.to(device)

    def batch_apply_move(self, game_states, actions):
        new_game_states = []
        for idx, game_state in enumerate(game_states):
            action_idx = actions[idx].item()
            decoded_move = self.encoder.decode_point_index(action_idx)
            move = Move.play(decoded_move)
            new_game_state = game_state.apply_move(move)
            new_game_states.append(new_game_state)
        return new_game_states

    def generate_rewards(self, rewards):
        for idx, r in enumerate(rewards):
            self.batched_rewards[idx] = r
        self.batched_rewards = self.batched_rewards.float().to(device)
        return self.batched_rewards

## 4 Plane Encoder

Original AlphaGo uses a 48 plane encoder, we use 4 plane encoder, which is also mentioned in the paper.

In [9]:
import numpy as np

"""
Feature name            num of planes   Description
Stone colour            3               Player stone / opponent stone / empty
Ones                    1               A constant plane filled with 1
"""

FEATURE_OFFSETS = {
    "stone_color": 0,
    "ones": 3,
    "current_player_color": 4,
    "legal_moves": 5
}


def offset(feature):
    return FEATURE_OFFSETS[feature]

# TODO: I need to change the Four plane encoder, just because I have changed the environment
class FourplaneEncoder:
    def __init__(self, board_size=(19, 19), use_player_plane=True, use_legal_moves=True):
        self.board_width, self.board_height = board_size
        self.use_player_plane = use_player_plane
        self.use_legal_moves = use_legal_moves
        self.num_planes = 4 + use_player_plane + use_legal_moves

    def name(self):
        return 'fourplane'

    def encode(self, game_state):
        board_tensor = np.zeros((self.num_planes, self.board_height, self.board_width))
        
        # Set empty cells plane (default to empty)
        board_tensor[offset("stone_color") + 2] = 1
        
        # Iterate over occupied points only (much faster for sparse boards)
        next_player = game_state.next_player
        opponent = next_player.other
        stone_color_offset = offset("stone_color")
        
        for point, go_string in game_state.board._grid.items():
            if go_string is None:
                continue
            r = point.row - 1
            c = point.col - 1
            if go_string.color == next_player:
                board_tensor[stone_color_offset][r][c] = 1
                board_tensor[stone_color_offset + 2][r][c] = 0  # Not empty
            elif go_string.color == opponent:
                board_tensor[stone_color_offset + 1][r][c] = 1
                board_tensor[stone_color_offset + 2][r][c] = 0  # Not empty
        
        # Set ones plane once (moved outside loop)
        board_tensor[offset("ones")] = 1
        
        # Set player plane once (moved outside loop)
        if self.use_player_plane and next_player == Player.black:
            board_tensor[offset("current_player_color")] = 1
        
        # Set legal moves - optimized inline computation to avoid expensive deep copies
        if self.use_legal_moves:
            if not game_state.is_over():
                legal_moves_offset = offset("legal_moves")
                board = game_state.board
                
                # Fast inline legal move checking without deep copies
                for row in range(1, board.num_rows + 1):
                    for col in range(1, board.num_cols + 1):
                        point = Point(row, col)
                        
                        # Fast check: point must be empty
                        if board._grid.get(point) is not None:
                            continue
                        
                        # Check self-capture without deep copy
                        has_liberty = False
                        would_capture = False
                        friendly_strings = []
                        
                        for neighbor in point.neighbors():
                            if not board.is_on_grid(neighbor):
                                continue
                            neighbor_string = board._grid.get(neighbor)
                            if neighbor_string is None:
                                has_liberty = True
                                break
                            elif neighbor_string.color == next_player:
                                friendly_strings.append(neighbor_string)
                            else:  # opponent string
                                if neighbor_string.num_liberties == 1:
                                    would_capture = True
                        
                        # If has liberty, not self-capture
                        if not has_liberty:
                            # Check if all friendly strings would have no liberties
                            if friendly_strings and all(s.num_liberties == 1 for s in friendly_strings):
                                if not would_capture:
                                    continue  # Self-capture, skip
                        
                        # Check ko violation - simplified heuristic for performance
                        # Full ko check requires deep copy, so we use a fast heuristic:
                        # If last move captured exactly one stone and we're trying to recapture
                        # at that same position, it's likely a ko violation
                        if would_capture and game_state.last_move and game_state.last_move.is_play:
                            # Check if we're trying to play at the last move position
                            # (which would be recapturing after a single-stone capture)
                            if point == game_state.last_move.point:
                                # Count how many stones we'd capture
                                captured_stones = 0
                                for neighbor in point.neighbors():
                                    if not board.is_on_grid(neighbor):
                                        continue
                                    neighbor_string = board._grid.get(neighbor)
                                    if neighbor_string and neighbor_string.color == opponent:
                                        if neighbor_string.num_liberties == 1:
                                            captured_stones += len(neighbor_string.stones)
                                # If capturing exactly one stone at last move position, likely ko
                                if captured_stones == 1:
                                    continue  # Likely ko violation, skip
                        
                        # Valid move - set it
                        r = row - 1
                        c = col - 1
                        board_tensor[legal_moves_offset][r][c] = 1

        return board_tensor

    def ones(self):
        return np.ones((1, self.board_height, self.board_width))

    def zeros(self):
        return np.zeros((1, self.board_height, self.board_width))

    def encode_point(self, point):
        return self.board_width * (point.row - 1) + (point.col - 1)

    def decode_point_index(self, index):
        row = index // self.board_width
        col = index % self.board_width
        return Point(row=row + 1, col=col + 1)

    def num_points(self):
        return self.board_width * self.board_height

    def shape(self):
        return self.num_planes, self.board_height, self.board_width


def create(board_size):
    return FourplaneEncoder(board_size)


In [10]:
class FastFourplaneEncoder:
    def __init__(self, board_size=(19, 19), use_player_plane=True, use_legal_moves=True):
        self.board_width, self.board_height = board_size
        self.use_player_plane = use_player_plane
        self.use_legal_moves = use_legal_moves
        self.num_planes = 4 + use_player_plane + use_legal_moves

    def name(self):
        return 'fourplane'

    def encode(self, game_state):
        board_tensor = np.zeros((self.num_planes, self.board_height, self.board_width))
        
        # Set empty cells plane (default to empty)
        board_tensor[offset("stone_color") + 2] = 1
        
        # Iterate over occupied points only (much faster for sparse boards)
        next_player = game_state.next_player
        opponent = 3 - next_player
        stone_color_offset = offset("stone_color")

        curr_board = game_state.board

        for r in range(curr_board.num_rows):
            for c in range(curr_board.num_cols):
                if curr_board.grid[r,c] == next_player:
                    board_tensor[stone_color_offset][r][c] = 1
                    board_tensor[stone_color_offset+2][r][c] = 0
                elif curr_board.grid[r,c] == opponent:
                    board_tensor[stone_color_offset+1][r][c] = 1
                    board_tensor[stone_color_offset+2][r][c] = 0
        
        board_tensor[offset("ones")] = 1
        
        # if the player is black i.e. 1
        if self.use_player_plane:
            board_tensor[offset("current_player_color")] = 1
        
        # Set legal moves - optimized inline computation to avoid expensive deep copies
        if self.use_legal_moves:
            legal_moves_offset = offset("legal_moves")
            legal_moves = game_state.legal_moves()
            for move in legal_moves:
                if move.is_play:
                    point = move.point
                    lr = point.row - 1
                    lc = point.col - 1
                    board_tensor[legal_moves_offset][lr][lc] = 1

        return board_tensor

    def ones(self):
        return np.ones((1, self.board_height, self.board_width))

    def zeros(self):
        return np.zeros((1, self.board_height, self.board_width))

    def encode_point(self, point):
        return self.board_width * (point.row - 1) + (point.col - 1)

    def decode_point_index(self, index):
        row = index // self.board_width
        col = index % self.board_width
        return Point(row=row + 1, col=col + 1)

    def num_points(self):
        return self.board_width * self.board_height

    def shape(self):
        return self.num_planes, self.board_height, self.board_width


def create(board_size):
    return FourplaneEncoder(board_size)


## Supervised Learning Policy Network

 - Convolution layer with rectifier non-linearities.
 - CNN with 13 layers
 - Final softmax applied to legal moves
 - Data has been split into 4:1 train to test ratio. (In AlphaGo it is 28:1)
 - Pass moves has been excluded from the dataset
 - All 8 reflections and rotations has been applied and precomputed. We randomly sample mini batch from the augmented sample
 - Asynchronous stochastic gradient descent to minimize the log likelihood
 - Learning parameter is initialized to 0.03 and halved every 30k steps. (In AlphaGo, learning rate is 0.003 and halved every 80 mil training steps)
 - mini batch size of 16 as mentioned in the paper
 - zero momentum

In [23]:
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.utils.checkpoint as checkpoint

class SLPolicyNetwork(nn.Module):
    def __init__(self, features=5, filters=192):
        super(SLPolicyNetwork, self).__init__()
        self.features = features
        self.filters = filters
        self.first_layer = nn.Conv2d(self.features, self.filters, kernel_size=5, stride=1, padding=2)

        self.hidden_layers = nn.ModuleList([
            nn.Conv2d(self.filters, self.filters, kernel_size=3, stride=1, padding=1)
            for _ in range(11)
        ])

        self.final_layer = nn.Conv2d(self.filters, 1, kernel_size=1, stride=1)

    def forward(self, x, action_mask):
        x = F.relu(self.first_layer(x))
        def run_hidden(x):
            for layer in self.hidden_layers:
                x = F.relu(layer(x))
            return x

        x = checkpoint.checkpoint(run_hidden, x, use_reentrant=False)
        # Logits are the unnormalized scores output by the network for each possible move before softmax
        policy_logits = self.final_layer(x)  # (batch, 1, board_size, board_size)

        # apply legal move mask
        # Reshape and apply log_softmax for NLLLoss compatibility as per Alphago paper
        batch_size = policy_logits.size(0)
        policy_logits = policy_logits.view(batch_size, -1)
        legal_move_mask = action_mask.view(batch_size, -1)
        # apply log softmax to only legal moves
        masked_policy_logits = torch.where(
            legal_move_mask.bool(),
            policy_logits,
            torch.full_like(policy_logits, float('-inf'))
        )
        log_probs = F.log_softmax(masked_policy_logits, dim=1)
        return log_probs

## Supervised Learning Policy Trainer

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

class SLPolicyTrainer:
    def __init__(self, model):
        # initialize hyperparams
        self.model = model
        self.optimizer = None
        self.criterion = None
        self.scheduler = None

    def initialize(self):
        self.optimizer = optim.SGD(
            params = self.model.parameters(),
            lr = 0.03
        )
        self.scheduler = StepLR(self.optimizer, step_size=30000, gamma=0.5)
        self.criterion = nn.NLLLoss()
        return
    def train(self, loader):
        self.model.train()
        total_loss = 0
        correct_predictions = 0
        total_sample = 0
        for (x, y) in loader:
            # Cast input to float32
            x = x.to(torch.float32)
            self.optimizer.zero_grad()
            log_probs = self.model(x)

            loss = self.criterion(log_probs, y)

            loss.backward()
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()
            total_sample += y.size(0)
            prediction = log_probs.argmax(dim=1)
            correct_predictions += (prediction == y).sum().item()

        avg_loss = total_loss / len(loader)
        acc = correct_predictions / total_sample

        return avg_loss, acc
    def evaluate(self, loader):
        self.model.eval()
        total_loss = 0
        total_sample = 0
        correct_predictions = 0
        with torch.no_grad():
            for (x, y) in loader:
                # Cast input to float32
                x = x.to(torch.float32)
                log_probs = self.model(x)
                loss = self.criterion(log_probs, y)
                prediction = log_probs.argmax(dim=1)
                total_loss += loss.item()
                total_sample += y.size(0)
                correct_predictions += (prediction == y).sum().item()


        avg_loss = total_loss / len(loader)
        acc = correct_predictions / total_sample
        return avg_loss, acc

## DataLoader for Go

In [ ]:
import numpy as np
import torch
import torch.utils.data as td

__all__ = [
    'GoDataLoader'
]

class GoDataLoader:
    def __init__(self, feature_path, label_path = None):
        self.feature_path = feature_path
        self.label_path = label_path

    def load_data(self):
        features = np.load(self.feature_path)
        features_tensor = torch.from_numpy(features)

        if self.label_path:
            labels = np.load(self.label_path)
            labels_tensor = torch.from_numpy(labels).to(torch.int64) # Cast labels to torch.int64
            # this dataset can be directly used with torch's DataLoader
            dataset = td.TensorDataset(features_tensor, labels_tensor)
        else:
            dataset = td.TensorDataset(features_tensor)
        return dataset

## Training Loop of Supervised Learning Policy

In [ ]:
from torch.utils.data import DataLoader
import torch
import torch.optim as optim
import torch.nn as nn

print("Load model")
model = SLPolicyNetwork(features=6)
num_of_epochs = 10

# prepare the dataset
print("fetch and load data")
training_feature_path = '/kaggle/input/alphago-kgs-200k/KGS-2019_04-19-1255-_train_features.npy'
training_label_path = '/kaggle/input/alphago-kgs-200k/KGS-2019_04-19-1255-_train_labels.npy'
test_feature_path = '/kaggle/input/alphago-kgs-200k/KGS-2019_04-19-1255-_test_features.npy'
test_label_path = '/kaggle/input/alphago-kgs-200k/KGS-2019_04-19-1255-_test_labels.npy'
training_dataset = GoDataLoader(training_feature_path, training_label_path).load_data()
test_dataset = GoDataLoader(test_feature_path, test_label_path).load_data()

# load the dataset to a loader
# this is what we will pass to our Trainer
training_loader = DataLoader(training_dataset, batch_size=16, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

# load the trainer
trainer = SLPolicyTrainer(model)
trainer.initialize()

for epoch in range(num_of_epochs):
    training_loss, training_acc = trainer.train(training_loader)
    test_loss, test_acc = trainer.evaluate(test_loader)
    print(f"epoch: {epoch}, training loss: {training_loss}, training acc: {training_acc}, test loss: {test_loss}, test_acc: {test_acc}")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': trainer.optimizer.state_dict(),
        'train_acc': training_acc,
        'test_acc': test_acc,
    }, f'sl_policy_epoch_{epoch}.pth')

In [12]:
# check if gpu is available

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

CUDA available: True
CUDA device count: 2
Current CUDA device: 0
CUDA device name: Tesla T4


## Supervised Learning Policy Agent

In [13]:
import torch

class SLPolicyAgent:
    def __init__(self, model, temperature=1.0):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        self.encoder = FastFourplaneEncoder()
        self.temperature = temperature

    def select_move(self, x, action_mask):
        x = x.to(self.device)
        action_mask = action_mask.to(self.device)
        log_probs = self.model(x, action_mask)
        if self.temperature != 1.0:
            log_probs = log_probs / self.temperature
            # re-normalise after temperature scaling
            import torch.nn.functional as F
            log_probs = F.log_softmax(log_probs, dim=1)

        probs = log_probs.exp()  # convert log-probs to probs for multinomial
        actions = torch.multinomial(probs, num_samples=1).squeeze(1)  # [batch]
        selected_log_probs = log_probs[
            torch.arange(log_probs.size(0), device=self.device), actions
        ]  # [batch]
        return actions, selected_log_probs
        

## Supervised Learning Policy Bot Self Play

This is to test our Supervised Learning Policy Network

In [45]:
import time

num_of_games = 16

game_states = []

for _ in range(num_of_games):
    game_state = FastGameState.new_game()
    game_states.append(game_state)

model_file = torch.load('/kaggle/input/sl-policy-epoch-9/pytorch/default/1/sl_policy_epoch_9.pth', map_location=device)
agent_model = SLPolicyNetwork(features = 6).to(device)
opponent_model = SLPolicyNetwork(features = 6).to(device)
agent_model.load_state_dict(model_file['model_state_dict'])
opponent_model.load_state_dict(model_file['model_state_dict'])

agent_model.eval()
opponent_model.eval()

agent = SLPolicyAgent(agent_model) #black
opponent = SLPolicyAgent(opponent_model) #white

next_player = 1
encoder = FastFourplaneEncoder()

rewards = []

with torch.no_grad():
    while True:
        batched_game_states = []
        batched_legal_moves = []

        if len(rewards) == num_of_games:
            break
        for idx, game_state in enumerate(game_states):
            if not game_state.is_over():
                legal_moves = game_state.legal_moves()
                if len(legal_moves) == 2:
                    move = Move.resign()
                    game_state = game_state.apply_move(move)
                    winner = game_state.winner()
                    print(winner)
                    reward = 1 if winner == 1 else -1
                    rewards.append(reward)
                else:
                    batched_game_states.append(game_state)
                    batched_legal_moves.append(legal_moves)
            else:
                winner = game_state.winner()
                print(winner)
                reward = 1 if winner == 1 else -1
                rewards.append(reward)     
        if len(batched_game_states) == 0:
            continue
        batched_game_helper = BatchedGameState(batch_size=len(batched_game_states))
        x, action_mask = batched_game_helper.generate_batched_input(batched_game_states, batched_legal_moves)
        if next_player == 1:
            batched_action, batched_log_prob = agent.select_move(x, action_mask)
            game_states = batched_game_helper.batch_apply_move(batched_game_states, batched_action)
            
        else:
            batched_action, batched_log_prob = opponent.select_move(x, action_mask)
            game_states = batched_game_helper.batch_apply_move(batched_game_states, batched_action)
    
        next_player = 3 - next_player

batched_game_helper = BatchedGameState(batch_size=num_of_games)
batched_rewards = batched_game_helper.generate_rewards(rewards)  
print(batched_rewards)

2
2
1
2
1
1
2
2
1
2
1
2
1
2
1
1
tensor([-1., -1.,  1., -1.,  1.,  1., -1., -1.,  1., -1.,  1., -1.,  1., -1.,
         1.,  1.], device='cuda:0')


## Reinforcement learning of Policy Network

 - We use the same network as for Supervised Learning
 - We initialize the policy to be theta ( which is the supervised learning model)
 - We also use an opponent pool to randomly select opponent for mini batch training as mentioned in the paper
 - Also, using baseline default to 0 for the first pass. AlphaGo uses value network as baseline in the second pass
 - 10k episodes of 128 mini batch size
 - We add a model to the opponent pool after every 500 episodes
 - Used REINFORCE algorithm with stochastic gradient ascent for policy updates.

In [37]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import torch.optim as optim
import torch
import random
import time

class RLPolicyTrainer:
    def __init__(self, baseline = 0.0):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = SLPolicyNetwork(features=6).to(self.device)
        self.opponent_pool = []
        self.optimizer = None
        self.baseline = baseline
        self.episodes = 20
        self.mini_batch = 16
        self.board_size = 19
        self.encoder = FastFourplaneEncoder()

    def initialize(self):
        model_file = torch.load('/kaggle/input/sl-policy-epoch-9/pytorch/default/1/sl_policy_epoch_9.pth', map_location=self.device)
        self.model.load_state_dict(model_file['model_state_dict'])
        self.optimizer = optim.SGD(params=self.model.parameters(), lr=0.0003)
        opponent_model = SLPolicyNetwork(features=6).to(self.device)
        opponent_model.load_state_dict(model_file['model_state_dict'])
        self.opponent_pool.append(opponent_model)

    def train_batch(self):
        opponent_model = random.choice(self.opponent_pool)
        opponent_model.to(self.device)
        opponent_model.eval()
        
        agent = SLPolicyAgent(self.model)
        opponent = SLPolicyAgent(opponent_model)
        
        total_wins = 0

        game_states = []
        for i in range(self.mini_batch):
            game_state = FastGameState.new_game(self.board_size)
            game_states.append(game_state)

        log_prob_sums = [torch.tensor(0.0, device=self.device, requires_grad=False)
                     for _ in range(self.mini_batch)]

        batched_game_helper = BatchedGameState(batch_size=self.mini_batch)

        next_player = 1
        while True:
            batched_game_states = []
            batched_legal_moves = []
            batched_game_idx = []

            for idx, game_state in enumerate(game_states):
                if not game_state.is_over():
                    legal_moves = game_state.legal_moves()
                    if len(legal_moves) == 2:
                        move = Move.resign()
                        game_state = game_state.apply_move(move)
                        game_states[idx] = game_state
                    else:
                        batched_game_states.append(game_state)
                        batched_legal_moves.append(legal_moves)
                        batched_game_idx.append(idx)
                
            if not batched_game_states:
                break

            active = len(batched_game_states)
            if active != batched_game_helper.batch_size:
                batched_game_helper = BatchedGameState(batch_size=active)

            x, action_mask = batched_game_helper.generate_batched_input(batched_game_states, batched_legal_moves)

            if next_player == 1:
                batched_action, batched_log_prob = agent.select_move(x, action_mask)
                new_game_states = batched_game_helper.batch_apply_move(batched_game_states, batched_action)
                for i, idx in enumerate(batched_game_idx):
                    log_prob_sums[idx] = log_prob_sums[idx] + batched_log_prob[i]
                    game_states[idx] = new_game_states[i]
                
            else:
                with torch.no_grad():
                    batched_action, batched_log_prob = opponent.select_move(x, action_mask)
                    new_game_states = batched_game_helper.batch_apply_move(batched_game_states, batched_action)
                    for i, idx in enumerate(batched_game_idx):
                        game_states[idx] = new_game_states[i]
            
            next_player = 3 - next_player

        raw_rewards = []
        for game_state in game_states:
            winner = game_state.winner()
            reward = 1 if winner == 1 else -1
            if reward == 1:
                total_wins += 1
            raw_rewards.append(reward - self.baseline)

        rewards = torch.tensor(raw_rewards, dtype=torch.float32, device=self.device)

        log_prob_sums_tensor = torch.stack(log_prob_sums)
        avg_loss = (-rewards * log_prob_sums_tensor).mean()
        win_rate = total_wins / self.mini_batch
        
        return avg_loss, win_rate

    def train(self):
        for i in range(self.episodes):
            start = time.time()
            print(f"starting episode: {i+1}")
            self.model.train()
            self.optimizer.zero_grad()
            expected_loss, win_rate = self.train_batch()
            loss_val = expected_loss.item()
            expected_loss.backward()
            self.optimizer.step()
            end = time.time()
            print(f"expected loss: {loss_val}, accuracy: {win_rate}, time taken to run one episode: {end-start}")
            
            if (i+1) % 100 == 0:
                opponent_model = SLPolicyNetwork(features=6).to(self.device)
                opponent_model.load_state_dict(self.model.state_dict())
                opponent_model.eval()
                self.opponent_pool.append(opponent_model)

            if (i + 1) % 100 == 0 or (i + 1) == self.episodes:
                torch.save({
                    'episode': i + 1,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'win_rate': win_rate,
                    'loss': loss_val,
                }, f'rl_policy_episode_{i+1}.pth')
                print(f"Saved checkpoint: rl_policy_episode_{i+1}.pth")
                
        
                    

rl_trainer = RLPolicyTrainer()
rl_trainer.initialize()
rl_trainer.train()
        
        
        

starting episode: 1
expected loss: -112.83578491210938, accuracy: 0.4375, time taken to run one episode: 87.25963544845581
starting episode: 3
expected loss: 141.9828643798828, accuracy: 0.5625, time taken to run one episode: 83.34611201286316
starting episode: 4
expected loss: -121.54344177246094, accuracy: 0.4375, time taken to run one episode: 79.80488324165344
starting episode: 5
expected loss: -247.2866668701172, accuracy: 0.375, time taken to run one episode: 82.77465987205505
starting episode: 6
expected loss: 412.8511047363281, accuracy: 0.6875, time taken to run one episode: 81.31649804115295
starting episode: 7
expected loss: 105.73518371582031, accuracy: 0.5625, time taken to run one episode: 79.93908429145813
starting episode: 8
expected loss: 298.0963134765625, accuracy: 0.625, time taken to run one episode: 77.70336651802063
starting episode: 9
expected loss: 141.077880859375, accuracy: 0.5625, time taken to run one episode: 82.13407135009766
starting episode: 10
expected

In [ ]:
class RandomPolicyAgent:
    def select_move(game_state = None, legal_moves):
        legal_play_moves = legal_moves[:-2]
        return random.choice(legal_play_moves)

## Reinforcement Learning of Value Network

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch

class ValueNetwork(nn.Module):
    def __init__(self, features=5, filters=192):
        super(ValueNetwork, self).__init__()
        self.features = features
        self.filters = filters
        self.first_layer = nn.Conv2d(self.features, self.filters, kernel_size=5, stride=1, padding=2)

        self.hidden_layers = nn.ModuleList([
            nn.Conv2d(self.filters, self.filters, kernel_size=3, stride=1, padding=1)
            for _ in range(12)
        ])

        self.thirteenth_layer = nn.Conv2d(self.filters, 1, kernel_size=1, stride=1)
        self.fourteenth_layer = nn.Linear(19*19, 256)

        self.final_layer = nn.Linear(256, 1)

    def forward(self, x, action_mask):
        x = F.relu(self.first_layer(x))
        def run_hidden(x):
            for layer in self.hidden_layers:
                x = F.relu(layer(x))
            return x

        run_hidden()
        x = F.relu(self.thirteenth_layer(x))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fourteenth_layer(x))
        x = F.Tanh(self.final_layer(x))

        return x
        
    

In [ ]:
import torch
import random
from torch.utils.data import TensorDataset, DataLoader

class ValueNetworkDataLoader:
    def __init__(self):
        self.batch_size = 16
        self.time_step = 2
    def generate_games(self):
        sl_policy_model_file = torch.load('/kaggle/input/sl-policy-epoch-9/pytorch/default/1/sl_policy_epoch_9.pth', map_location=device)
        sl_policy_model = SLPolicyNetwork(features = 6).to(device)
        sl_policy_model.load_state_dict(model_file['model_state_dict'])
        sl_policy_model.eval()
        self.sl_policy_agent = SLPolicyAgent(sl_policy_model)

        self.random_policy_agent = RandomPolicyAgent()

        rl_policy_model_file = torch.load('/kaggle/input/models/yashasvaparas/rl-trained-policy-20-epoch/pytorch/default/1/rl_policy_episode_20.pth', map_location=device)
        rl_policy_model_file = SLPolicyNetwork(features = 6).to(device)
        rl_policy_model_file.load_state_dict(model_file['model_state_dict'])
        rl_policy_model_file.eval()
        self.rl_policy_agent = SLPolicyAgent(rl_policy_model_file)

        # generate over 1000 games
        self.episodes = 63

        all_features = []
        all_labels   = []

        for episode in range(self.episodes):
            features, labels = self.play_batch()
            all_features.append(features)
            all_labels.append(labels)

        all_features = torch.cat(all_features, dim=0)  # (N, C, H, W)
        all_labels   = torch.cat(all_labels,   dim=0)  # (N,)

        dataset    = TensorDataset(all_features, all_labels)
        dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        return dataloader


    def play_batch(self):
        game_states = [FastGameState.new_game() for _ in range(self.batch_size)]
        # instead of choosing a split for each game, we choose the splitting index
        # for the whole batch
        u = random.randint(2, 449)

        next_player = 1
        rewards = torch.empty(self.batch_size)
        step_count = 1
        captured_x  = None
        last_x = None

        with torch.no_grad():
            while not all(gs.is_over() for gs in game_states):
                batched_legal_moves = [gs.legal_moves() for gs in game_states]
        
                for idx, (gs, legal_moves) in enumerate(zip(game_states, batched_legal_moves)):
                    if len(legal_moves) <= 2:
                        game_states[idx] = gs.apply_move(Move.resign())

                batched_game_states = [gs for gs in game_states if not gs.is_over()]

                batched_game_helper = BatchedGameState(batch_size=len(batched_game_states))
                x, action_mask = batched_game_helper.generate_batched_input(
                    batched_game_states, batched_legal_moves
                )
                if step_count == u + 1:
                    captured_x = x
                last_x = x
                agent = self.select_agent(u, step_count)
                batched_action, batched_log_prob = agent.select_move(x, action_mask)
                game_states = batched_game_helper.batch_apply_move(batched_game_states, batched_action)
                next_player = 3 - next_player
                step_count += 1
        for idx, gs in enumerate(game_states):
        winner = gs.winner()
        rewards[idx] = 1.0 if winner == 1 else -1.0

        if captured_x is None:
            captured_x = last_x

        return captured_x, rewards

    def select_agent(u, step_count):
        if u < step_count:
            return self.sl_policy_agent
        if u == step_count:
            return self.random_policy_agent
        if u > step_count:
            return self.rl_policy_agent


        
        

        
        